# Crop Disease Detection — Model 2 (MobileNetV2) – Colab version

**Goal:** a second model for the same app, built to do **both** jobs at once, so no choice is needed first:
1. **A different architecture for viva comparison** — MobileNetV2 vs the first model's EfficientNetV2B0.
2. **Better field-photo robustness** — stronger augmentation (rotation / zoom / contrast / shift) that simulates the messy light and framing of Google/field photos, which is exactly where Model 1 went wrong.

**Drop-in swap (by design):** same dataset, same 29 classes in the same order, same `IMG_SIZE = 192`, same `SEED = 42`, and the MobileNetV2 preprocessing (`[-1, 1]` scaling) is built **inside** the saved model — so the app needs **no** change: same `ml-service/classes.json`, same `INPUT_SIZE=192`, same `PREPROCESS_SCALE=none`. Just copy the new `.keras` file in and restart the ML service.

**Before running:** `Runtime → Change runtime type → T4 GPU`, then run the cells top to bottom. Expect roughly 30–45 minutes on a T4 (MobileNetV2 is lighter than EfficientNetV2B0; Model 1 took ~55 minutes).


In [ ]:
# 1. Check GPU + download dataset from Kaggle (public dataset, no API key needed)
!nvidia-smi
import kagglehub
dataset_path = kagglehub.dataset_download('snikhilrao/crop-disease-detection-dataset')
print("Dataset downloaded to:", dataset_path)


In [ ]:
# 2. Imports
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt

print("TensorFlow version:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices('GPU'))


In [ ]:
# 3. Locate the Train / Val / Test folders automatically
#    (the folder layout inside the Kaggle download differs from /kaggle/input)
def find_splits(root):
    for dirpath, dirnames, _ in os.walk(root):
        lower = {d.lower(): d for d in dirnames}
        val_key = next((k for k in ('val', 'valid', 'validation') if k in lower), None)
        if 'train' in lower and val_key and 'test' in lower:
            return (os.path.join(dirpath, lower['train']),
                    os.path.join(dirpath, lower[val_key]),
                    os.path.join(dirpath, lower['test']))
    raise FileNotFoundError(f"Could not find Train/Val/Test folders under {root}")

train_dir, val_dir, test_dir = find_splits(dataset_path)
print("Train:", train_dir)
print("Val  :", val_dir)
print("Test :", test_dir)
print("Number of train classes:", len(os.listdir(train_dir)))


In [ ]:
# 4. Data pipeline — SAME splits / size / seed as Model 1, STRONGER augmentation
#    Same IMG_SIZE (192) and SEED (42) as Model 1, so class order and app settings match.
#    Stronger augmentation is the field-robustness part: it simulates varied light,
#    angle, framing and contrast, which lab-style PlantVillage photos do not have.
IMG_SIZE = 192
BATCH_SIZE = 32
SEED = 42
AUTOTUNE = tf.data.AUTOTUNE

data_augmentation = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.2),        # was 0.1 in Model 1
    layers.RandomZoom(0.2),            # was 0.1 in Model 1
    layers.RandomContrast(0.2),        # NEW — simulates varied field lighting
    layers.RandomTranslation(0.1, 0.1) # NEW — simulates off-centre field framing
])

def prepare_dataset(directory, augment=False, shuffle=False):
    ds = keras.utils.image_dataset_from_directory(
        directory,
        seed=SEED,
        image_size=(IMG_SIZE, IMG_SIZE),
        batch_size=BATCH_SIZE,
        label_mode='categorical',
        shuffle=shuffle,          # shuffle only training data
    )
    class_names = ds.class_names
    if augment:
        # augmentation is applied fresh every epoch (not cached)
        ds = ds.map(lambda x, y: (data_augmentation(x, training=True), y),
                    num_parallel_calls=AUTOTUNE)
    return ds.prefetch(AUTOTUNE), class_names

train_ds, class_names = prepare_dataset(train_dir, augment=True, shuffle=True)
val_ds, _  = prepare_dataset(val_dir)
test_ds, _ = prepare_dataset(test_dir)   # shuffle=False so labels line up with predictions

NUM_CLASSES = len(class_names)
INV_CLASS_MAP = {i: name for i, name in enumerate(class_names)}
print("Classes:", NUM_CLASSES)
assert NUM_CLASSES == 29, f"Expected 29 classes, got {NUM_CLASSES} — check the dataset folders"
print(class_names)


In [ ]:
# 5. Model (MobileNetV2 transfer learning) — preprocessing INSIDE the model
#    MobileNetV2 expects inputs scaled to [-1, 1]. Instead of scaling in the
#    data pipeline (which the app would then have to repeat), the Rescaling
#    layer below is part of the saved model — so the app can keep feeding it
#    raw 0-255 pixels with PREPROCESS_SCALE=none, exactly like Model 1.
from tensorflow.keras import mixed_precision
from tensorflow.keras.applications import MobileNetV2

mixed_precision.set_global_policy('mixed_float16')   # T4 GPU supports this

def build_model(num_classes, img_size=IMG_SIZE):
    base = MobileNetV2(include_top=False,
                       input_shape=(img_size, img_size, 3),
                       weights='imagenet')
    base.trainable = False   # Phase 1: backbone frozen

    inputs = keras.Input(shape=(img_size, img_size, 3))
    x = layers.Rescaling(1./127.5, offset=-1)(inputs)  # 0-255 -> [-1, 1], inside the model
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.4)(x)
    outputs = layers.Dense(num_classes, activation='softmax', dtype='float32')(x)  # float32 output for mixed precision
    return keras.Model(inputs, outputs), base

model, base_model = build_model(NUM_CLASSES)
model.compile(optimizer=keras.optimizers.Adam(1e-4),
              loss='categorical_crossentropy',
              metrics=['accuracy'])
model.summary()


In [ ]:
# 6. Two-phase training (same recipe as Model 1, so the comparison is fair)
EPOCHS_PHASE1 = 2
EPOCHS_PHASE2 = 6
CHECKPOINT_PATH = 'plant_mobilenetv2_best.keras'

def make_callbacks():
    return [
        keras.callbacks.ModelCheckpoint(CHECKPOINT_PATH, save_best_only=True,
                                        monitor='val_accuracy', mode='max'),
        keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                          patience=2, min_lr=1e-7),
        keras.callbacks.EarlyStopping(monitor='val_loss', patience=3,
                                      restore_best_weights=True),
    ]

# ---- Phase 1: train only the classifier head ----
print("---- Phase 1: training top layers only ----")
history1 = model.fit(train_ds, validation_data=val_ds,
                     epochs=EPOCHS_PHASE1, callbacks=make_callbacks())

# ---- Phase 2: fine-tune the last 30 backbone layers (BatchNorm stays frozen) ----
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False
for layer in base_model.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

model.compile(optimizer=keras.optimizers.Adam(1e-5),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

print("---- Phase 2: fine-tuning last 30 layers ----")
history2 = model.fit(train_ds, validation_data=val_ds,
                     epochs=EPOCHS_PHASE2, callbacks=make_callbacks())


In [ ]:
# 7. Evaluate on the test set — compare with Model 1: 96.10% accuracy, loss 0.1132
test_loss, test_acc = model.evaluate(test_ds)
print(f"Test Accuracy: {test_acc*100:.2f}%")
print(f"Test Loss: {test_loss:.4f}")
print("Model 1 (EfficientNetV2B0) for comparison: Test Accuracy 96.10%, Test Loss 0.1132")


In [ ]:
# 8. Sample predictions from the first test batch
for images, labels in test_ds.take(1):
    preds = model.predict(images, verbose=0)
    predicted_class = tf.argmax(preds, axis=1).numpy()
    true_class = tf.argmax(labels, axis=1).numpy()
    for i in range(len(images)):
        print(f"True: {INV_CLASS_MAP[int(true_class[i])]} | Predicted: {INV_CLASS_MAP[int(predicted_class[i])]}")


In [ ]:
# 9. Training curves
acc = history1.history['accuracy'] + history2.history['accuracy']
val_acc = history1.history['val_accuracy'] + history2.history['val_accuracy']
loss = history1.history['loss'] + history2.history['loss']
val_loss = history1.history['val_loss'] + history2.history['val_loss']

epochs = range(1, len(acc) + 1)

plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 1)
plt.plot(epochs, acc, label='Training Accuracy')
plt.plot(epochs, val_acc, label='Validation Accuracy')
plt.title('Training & Validation Accuracy'); plt.xlabel('Epochs'); plt.ylabel('Accuracy'); plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epochs, loss, label='Training Loss')
plt.plot(epochs, val_loss, label='Validation Loss')
plt.title('Training & Validation Loss'); plt.xlabel('Epochs'); plt.ylabel('Loss'); plt.legend()
plt.show()


In [ ]:
# 10. Confusion matrix + classification report
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

y_true = np.argmax(np.concatenate([y.numpy() for _, y in test_ds], axis=0), axis=1)
y_pred = np.argmax(model.predict(test_ds), axis=1)

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=False, cmap='Blues')
plt.xlabel('Predicted'); plt.ylabel('True'); plt.title('Confusion Matrix')
plt.show()

print("Classification Report:\n")
print(classification_report(y_true, y_pred, target_names=class_names))


In [ ]:
# 11. Save the model and download it to your computer
model.save('plant_mobilenetv2_final.keras')

try:
    from google.colab import files
    files.download('plant_mobilenetv2_final.keras')
except ImportError:
    pass
# Or save to Google Drive instead:
# from google.colab import drive
# drive.mount('/content/drive')
# model.save('/content/drive/MyDrive/plant_mobilenetv2_final.keras')


## Plug it into the app (2 minutes, no code change)

1. Copy `plant_mobilenetv2_final.keras` into `crop-disease-detection/ml-service/model/` (keep Model 1's file safe elsewhere — don't overwrite it until you've compared).
2. Restart the ML service (`python app.py`) — startup should print `✅ Model loaded: ...plant_mobilenetv2_final.keras`.
3. **Nothing else changes:** `INPUT_SIZE` stays `192`, `PREPROCESS_SCALE` stays `none`, `classes.json` stays as-is — the preprocessing is inside this model and the class order is the same.
4. Check `http://localhost:8000/health` shows `"modelLoaded": true`, then test the **same Google leaf photo** that Model 1 got wrong.

## Honest limitation (good viva line)

Stronger augmentation helps with varied light/framing, but it cannot fully fix the lab-photo → field-photo domain shift — the training photos are still lab-style. If field photos still fail after this model, the real fix is fine-tuning on actual field photos (e.g. a PlantDoc-style field dataset), not a third architecture on the same data.


In [ ]:
# 12. OPTIONAL — test field photos here in Colab, before swapping into the app
#     Upload 1-3 Google/field leaf photos when prompted, and see this model's Top-3.
#     If you also upload plant_effnetv2b0_final.keras, it prints Model 1's Top-3
#     next to Model 2's, so you can compare on the exact photo that failed.
import io
from PIL import Image

def top3(preds, label):
    idx = np.argsort(preds)[::-1][:3]
    print(f"  {label}:")
    for i in idx:
        print(f"    {INV_CLASS_MAP[int(i)]} — {preds[i]*100:.1f}%")

try:
    from google.colab import files as colab_files
    uploaded = colab_files.upload()
    for fname in uploaded:
        img = Image.open(io.BytesIO(uploaded[fname])).convert('RGB').resize((IMG_SIZE, IMG_SIZE))
        arr = np.expand_dims(np.array(img), axis=0)  # raw 0-255, like the app sends
        print(f"\n{fname}:")
        top3(model.predict(arr, verbose=0)[0], "Model 2 (MobileNetV2)")
        # Optional Model 1 comparison — upload plant_effnetv2b0_final.keras too:
        # model1 = tf.keras.models.load_model('plant_effnetv2b0_final.keras')
        # top3(model1.predict(arr, verbose=0)[0], "Model 1 (EfficientNetV2B0)")
except ImportError:
    print("Not running in Colab — skip this cell, or test the photo in the app instead.")
